In [1]:
import subprocess
subprocess.run(["pip", "install", "opencv-python-headless", "-q"], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, csv, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import scipy

DRIVE_ROOT = Path("/content/drive/MyDrive/suture v4")
sys.path.insert(0, str(DRIVE_ROOT))

from suture_geometry_v5 import (
    SutureGeometry, generate_dataset, GEO_TYPES, R_MIN_FRAC)

DIRS = {
    "data"  : DRIVE_ROOT / "data_v5",
    "plots" : DRIVE_ROOT / "plots_v5",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "specimen_size": 10.0,
    "random_seed"  : 42,
    "dataset_plan" : {
        "jigsaw"    : 300,
        "irregular" : 300,
        "sinusoidal": 200,
        "freeform"  : 200,
    },
    "N_range"    : (8, 14),
    "k_values"   : (2, 3, 4),
    "alpha_range": (0.20, 0.40),
}

np.random.seed(CONFIG["random_seed"])
random.seed(CONFIG["random_seed"])

print("=" * 55)
print("  suture_geometry_v5  —  4 types ready")
print("=" * 55)
print(f"  numpy {np.__version__}  scipy {scipy.__version__}")
print(f"  opencv {cv2.__version__}  pandas {pd.__version__}")
print(f"  Drive root : {DRIVE_ROOT}")
print(f"  Types      : {GEO_TYPES}")
print(f"  R_min threshold : {R_MIN_FRAC} * S = {R_MIN_FRAC*10:.3f} mm")
print("=" * 55)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  suture_geometry_v5  —  4 types ready
  numpy 2.0.2  scipy 1.16.3
  opencv 4.13.0  pandas 2.2.2
  Drive root : /content/drive/MyDrive/suture v4
  Types      : ['sinusoidal', 'jigsaw', 'irregular', 'freeform']
  R_min threshold : 0.01 * S = 0.100 mm


## 2 · Morphology ShowcaseDisplays one validated geometry of each type. **Top row:** geometry plot with control points (red). **Bottom row:** the 256×256 binary image used as CNN input.

In [2]:
test_cfg = {
    "sinusoidal": dict(N=9,  k=3, alpha=0.28),
    "jigsaw"    : dict(N=10, k=3, alpha=0.33),
    "irregular" : dict(N=12, k=4, alpha=0.30),
    "freeform"  : dict(N=10, k=3, alpha=0.28),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.patch.set_facecolor('#f8f8f8')

for col, gtype in enumerate(GEO_TYPES):
    cfg    = test_cfg[gtype]
    result = None
    for seed in range(30):
        geo = SutureGeometry(geo_type=gtype,
                             specimen_size=CONFIG["specimen_size"], **cfg)
        r   = geo.generate(seed=seed)
        if r["valid"]:
            result = r
            break

    if result is None:
        print(f"WARNING: {gtype} — no valid geometry in 30 seeds")
        continue

    ppath = f"/tmp/showcase_{gtype}.png"
    geo.visualize(result, save_path=ppath)

    axes[0, col].imshow(mpimg.imread(ppath))
    axes[0, col].set_title(
        f"{gtype.capitalize()}\n"
        f"N={result['N']}  k={result['k']}  alpha={result['alpha']:.2f}",
        fontsize=9, fontweight='bold', pad=3)
    axes[0, col].axis("off")

    axes[1, col].imshow(result["image"], cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title(
        f"CNN input (256x256)\nR_min={result['R_min']:.3f} mm",
        fontsize=8, pad=2)
    axes[1, col].axis("off")

plt.suptitle(
    "4 geometry types  |  B-spline family  |  physics-validated",
    fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
save_path = str(DIRS["plots"] / "4types_showcase.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

Saved -> /content/drive/MyDrive/suture v4/plots_v5/4types_showcase.png


## 3 · Effect of N — Control-Point Count`N` controls geometric **complexity**. More control points produce more interlocking teeth. Here `alpha` and `k` are fixed while `N` varies.

In [3]:
gtype  = "jigsaw"
alpha  = 0.33
N_vals = [6, 8, 10, 12, 14]

fig, axes = plt.subplots(2, 5, figsize=(24, 10))

for col, N in enumerate(N_vals):
    result = None
    for seed in range(20):
        geo = SutureGeometry(geo_type=gtype, N=N, k=3, alpha=alpha,
                             specimen_size=CONFIG["specimen_size"])
        r   = geo.generate(seed=seed)
        if r["valid"]:
            result = r
            break

    ppath = f"/tmp/N_{N}.png"
    geo.visualize(result, save_path=ppath)
    n_teeth = max(2, N // 6)
    axes[0, col].imshow(mpimg.imread(ppath))
    axes[0, col].set_title(f"N={N}  (~{n_teeth} teeth)",
                           fontsize=10, fontweight='bold')
    axes[0, col].axis("off")
    axes[1, col].imshow(result["image"], cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title(f"arc={result['arc_length']:.2f} mm", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(
    f"Effect of N (control points)  |  type={gtype}  alpha={alpha}  k=3",
    fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
save_path = str(DIRS["plots"] / "N_effect.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

Saved -> /content/drive/MyDrive/suture v4/plots_v5/N_effect.png


## 4 · Effect of k — Spline Order`k` controls **smoothness**. The same control points are interpolated with different spline orders: `k=2` (quasi-linear), `k=3` (cubic), `k=4` (quartic). Higher `k` smooths the curve.

In [4]:
gtype  = "jigsaw"
alpha  = 0.33
N      = 10
k_vals = [2, 3, 4]

fig, axes = plt.subplots(2, 3, figsize=(14, 10))

for col, k in enumerate(k_vals):
    result = None
    for seed in range(20):
        geo = SutureGeometry(geo_type=gtype, N=N, k=k, alpha=alpha,
                             specimen_size=CONFIG["specimen_size"])
        r   = geo.generate(seed=seed)
        if r["valid"]:
            result = r
            break

    ppath = f"/tmp/k_{k}.png"
    geo.visualize(result, save_path=ppath)
    label = {2: "k=2  (quasi-linear)",
             3: "k=3  (cubic)",
             4: "k=4  (quartic)"}[k]
    axes[0, col].imshow(mpimg.imread(ppath))
    axes[0, col].set_title(label, fontsize=10, fontweight='bold')
    axes[0, col].axis("off")
    axes[1, col].imshow(result["image"], cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title(f"R_min={result['R_min']:.3f} mm", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(
    f"Effect of k (spline order)  |  type={gtype}  N={N}  alpha={alpha}",
    fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
save_path = str(DIRS["plots"] / "k_effect.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

Saved -> /content/drive/MyDrive/suture v4/plots_v5/k_effect.png


## 5 · Effect of alpha — Amplitude`alpha` controls the **tooth depth** (interlocking amplitude). Larger `alpha` produces deeper teeth and a stronger mechanical lock.

In [5]:
gtype      = "jigsaw"
alpha_vals = [0.15, 0.22, 0.30, 0.40]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for col, alpha in enumerate(alpha_vals):
    result = None
    for seed in range(20):
        geo = SutureGeometry(geo_type=gtype, N=10, k=3, alpha=alpha,
                             specimen_size=CONFIG["specimen_size"])
        r   = geo.generate(seed=seed)
        if r["valid"]:
            result = r
            break

    ppath = f"/tmp/alpha_{alpha}.png"
    geo.visualize(result, save_path=ppath)
    axes[0, col].imshow(mpimg.imread(ppath))
    axes[0, col].set_title(f"alpha = {alpha}", fontsize=11, fontweight='bold')
    axes[0, col].axis("off")
    axes[1, col].imshow(result["image"], cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title("CNN input", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(
    f"Effect of alpha (amplitude)  |  type={gtype}  N=10  k=3",
    fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
save_path = str(DIRS["plots"] / "alpha_effect.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

Saved -> /content/drive/MyDrive/suture v4/plots_v5/alpha_effect.png


## 6 · Full Dataset GenerationGenerates the complete **1,000-geometry dataset**. For each geometry this saves:- a **256×256 binary image** (CNN input)- a **CSV row** (parameters + placeholders for FEA results)- an **Abaqus-ready JSON** (`curve_points` + metadata)All JSON files are collected into a single shared folder (`all_json`) so they can be copied to the lab PC in one step. *Takes ~3–5 minutes.*

In [6]:
ALL_JSON_DIR = DIRS["data"] / "all_json"
ALL_JSON_DIR.mkdir(parents=True, exist_ok=True)

all_csv_paths = []
total_saved   = 0

print("Generating 1,000 suture geometries (+ Abaqus JSON each) ...\n")
print(f"{'Type':<14} {'Target':>7} {'Saved':>7}")
print("-" * 32)

for gtype, n_target in CONFIG["dataset_plan"].items():
    out_dir  = str(DIRS["data"] / gtype)
    csv_path = generate_dataset(
        output_dir    = out_dir,
        geo_type      = gtype,
        n_samples     = n_target,
        N_range       = CONFIG["N_range"],
        k_values      = CONFIG["k_values"],
        alpha_range   = CONFIG["alpha_range"],
        specimen_size = CONFIG["specimen_size"],
        seed          = CONFIG["random_seed"],
        skip_invalid  = True,
        verbose       = False,
        export_json   = True,
        json_dir      = str(ALL_JSON_DIR),
    )
    all_csv_paths.append(csv_path)
    saved = len(pd.read_csv(csv_path))
    total_saved += saved
    print(f"  {gtype:<12} {n_target:>7} {saved:>7}")

master_df  = pd.concat(
    [pd.read_csv(p) for p in all_csv_paths], ignore_index=True)
master_csv = str(DIRS["data"] / "master_dataset.csv")
master_df.to_csv(master_csv, index=False)

n_json = len(list(ALL_JSON_DIR.glob("*.json")))

print("-" * 32)
print(f"  {'TOTAL':<12} "
      f"{sum(CONFIG['dataset_plan'].values()):>7} {total_saved:>7}")
print(f"\nMaster CSV -> {master_csv}")
print(f"JSON files -> {ALL_JSON_DIR}  ({n_json} files)")
print(f"Valid rate : {master_df['valid'].mean()*100:.1f}%")

Generating 1,000 suture geometries (+ Abaqus JSON each) ...

Type            Target   Saved
--------------------------------
  jigsaw           300     300
  irregular        300     300
  sinusoidal       200     200
  freeform         200     200
--------------------------------
  TOTAL           1000    1000

Master CSV -> /content/drive/MyDrive/suture v4/data_v5/master_dataset.csv
JSON files -> /content/drive/MyDrive/suture v4/data_v5/all_json  (1000 files)
Valid rate : 100.0%


## 7 · Package JSONs for AbaqusZips every geometry JSON into a single archive for transfer to the lab PC.**Next steps after this cell:**1. Download `all_geometries_json.zip` from Drive (`data_v5` folder)2. Unzip into `C:\temp\Scraba\geometries\` on the lab PC3. Run `abaqus python run_batch.py`

In [7]:
import shutil
zip_base = str(DIRS["data"] / "all_geometries_json")
shutil.make_archive(zip_base, 'zip', str(ALL_JSON_DIR))
print(f"Zipped -> {zip_base}.zip")
print(f"Contains {len(list(ALL_JSON_DIR.glob('*.json')))} JSON files")
print("\nNext steps:")
print("  1. Download all_geometries_json.zip from Drive")
print("  2. Unzip into C:\\temp\\Scraba\\geometries\\ on the lab PC")
print("  3. Run:  abaqus python run_batch.py")

Zipped -> /content/drive/MyDrive/suture v4/data_v5/all_geometries_json.zip
Contains 1000 JSON files

Next steps:
  1. Download all_geometries_json.zip from Drive
  2. Unzip into C:\temp\Scraba\geometries\ on the lab PC
  3. Run:  abaqus python run_batch.py


## 8 · Dataset InspectionSummary statistics and distribution plots across the generated dataset: count per type, spline-order distribution, amplitude distribution, and minimum curvature radius.

In [8]:
df = pd.read_csv(str(DIRS["data"] / "master_dataset.csv"))

print(f"Total: {len(df)}  |  Valid: {df['valid'].sum()}  "
      f"({df['valid'].mean()*100:.1f}%)\n")

print(df.groupby("geo_type")[
    ["valid", "N", "alpha", "R_min", "arc_length"]
].agg({
    "valid"     : "sum",
    "N"         : "mean",
    "alpha"     : "mean",
    "R_min"     : "mean",
    "arc_length": "mean",
}).round(3).to_string())

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
colors = ["#1a5fb4", "#2ec27e", "#e5a50a", "#9141ac"]

counts = df["geo_type"].value_counts().reindex(GEO_TYPES)
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title("Count per type", fontsize=10)
axes[0].tick_params(axis='x', rotation=20)

df["k"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color="#1a5fb4", edgecolor="white")
axes[1].set_title("k distribution", fontsize=10)
axes[1].set_xlabel("spline order k")
axes[1].tick_params(axis='x', rotation=0)

df["alpha"].hist(ax=axes[2], bins=25, color="#2ec27e",
                  edgecolor="white", lw=0.4)
axes[2].set_title("alpha distribution", fontsize=10)
axes[2].set_xlabel("alpha")

df["R_min"].hist(ax=axes[3], bins=30, color="#e5a50a",
                  edgecolor="white", lw=0.4)
axes[3].set_title("R_min distribution (mm)", fontsize=10)
axes[3].set_xlabel("min curvature radius (mm)")

plt.suptitle("Dataset statistics - physics-validated geometries",
             fontsize=11, y=1.02)
plt.tight_layout()
save_path = str(DIRS["plots"] / "dataset_statistics.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

Total: 1000  |  Valid: 1000  (100.0%)

            valid       N  alpha  R_min  arc_length
geo_type                                           
freeform      200  10.780  0.293  0.232      16.782
irregular     300  10.800  0.297  0.274      16.819
jigsaw        300  11.077  0.302  0.452      17.849
sinusoidal    200  11.150  0.300  0.679      17.405
Saved -> /content/drive/MyDrive/suture v4/plots_v5/dataset_statistics.png


## 9 · Design-Space GalleryA 4×4 grid sampling the design space across all types and randomized `N`, `k`, `alpha` values — a visual overview of the dataset's diversity.

In [9]:
rng_g = np.random.default_rng(7)
fig, axes = plt.subplots(4, 4, figsize=(20, 20))

for idx, ax in enumerate(axes.flat):
    gtype = GEO_TYPES[idx % len(GEO_TYPES)]
    alpha = float(rng_g.uniform(0.20, 0.40))
    N     = int(rng_g.integers(8, 15))
    k     = int(rng_g.choice([2, 3, 4]))

    result = None
    for seed in range(20):
        geo = SutureGeometry(geo_type=gtype, N=N, k=k, alpha=alpha,
                             specimen_size=CONFIG["specimen_size"])
        r   = geo.generate(seed=seed)
        if r["valid"]:
            result = r
            break

    ppath = f"/tmp/gallery_{idx}.png"
    geo.visualize(result, save_path=ppath)
    ax.imshow(mpimg.imread(ppath))
    ax.set_title(f"{gtype}  N={N}  k={k}  alpha={alpha:.2f}", fontsize=7)
    ax.axis("off")

plt.suptitle(
    "Design space gallery - 4 types, diverse N, k, alpha",
    fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
save_path = str(DIRS["plots"] / "gallery.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

Saved -> /content/drive/MyDrive/suture v4/plots_v5/gallery.png


## 10 · Single-Geometry Abaqus Bridge *(optional)*Exports **one** example geometry JSON for testing a single Abaqus run. The full dataset JSONs are already produced in Section 6 — this cell is only for quick single-geometry validation.

In [10]:
geo    = SutureGeometry(geo_type="jigsaw", N=10, k=3, alpha=0.33,
                        specimen_size=CONFIG["specimen_size"])
result = None
for seed in range(20):
    r = geo.generate(seed=seed)
    if r["valid"]:
        result = r
        break

cp = result["curve_points"]

print("Abaqus-ready curve_points")
print("=" * 55)
print(f"  geo_type   : {result['geo_type']}")
print(f"  N          : {result['N']}")
print(f"  k          : {result['k']}")
print(f"  alpha      : {result['alpha']}")
print(f"  R_min      : {result['R_min']} mm")
print(f"  arc_length : {result['arc_length']} mm")
print(f"  num points : {len(cp)}")

json_path = str(DIRS["data"] / "example_abaqus_input.json")
with open(json_path, "w") as f:
    json.dump({
        "geo_type"    : result["geo_type"],
        "N"           : result["N"],
        "k"           : result["k"],
        "alpha"       : result["alpha"],
        "R_min"       : result["R_min"],
        "arc_length"  : result["arc_length"],
        "curve_points": cp,
    }, f, indent=2)
print(f"\n  JSON saved -> {json_path}")

Abaqus-ready curve_points
  geo_type   : jigsaw
  N          : 10
  k          : 3
  alpha      : 0.33
  R_min      : 0.41675 mm
  arc_length : 17.60057 mm
  num points : 800

  JSON saved -> /content/drive/MyDrive/suture v4/data_v5/example_abaqus_input.json
